# 07 Apply Views — Orchestrator

**Doel:** Achter elkaar uitvoeren van alle view-notebooks in `views/integration/` en
`views/datamart/`. Views kunnen niet binnen DLT bestaan (DLT beheert
Materialised Views en Streaming Tables, geen plain views), dus elke view is een
eigen notebook met één `%sql CREATE OR REPLACE VIEW` cel. Deze orchestrator
roept ze op met `dbutils.notebook.run()` en geeft de `catalog`-widget door.

## Volgorde

1. `integration/sales_line` — eerst, want `FCT_SALES_LINE` MV (downstream in
   `dlt_datamart`) leest hiervan.
2. `datamart/dim_*` — 9 dim-notebooks, geen onderlinge afhankelijkheden.

```
setup
 ├─→ ingest_full
 └─→ ingest_incremental
       ↓
       dlt_integration                  (Streaming Tables in INTEGRATION)
        ↓
        apply_views                     (deze notebook — fan-out naar 10 view-notebooks)
         ↓
         dlt_datamart                   (FCT_ORDER MV + FCT_SALES_LINE MV — leest SALES_LINE view)
```

## Patroon-overzicht in de stack

| Object | Patroon | Reden |
|---|---|---|
| `STAGING_AZURESTORAGE.*` | Delta Table | Landing zone, CDF, Time Travel |
| `INTEGRATION.DWH_ORDER_HEADER` / `DWH_ORDER_DETAIL` | Streaming Table | apply_changes state, MERGE |
| `INTEGRATION.SALES_LINE` | **View** (eigen notebook) | Pure join, geen state, altijd vers |
| `DATAMART.DIM_*` (9 stuks) | **View** (eigen notebook per dim) | Lage cardinaliteit, simpele projecties |
| `DATAMART.FCT_ORDER` | MV (in `dlt_datamart`) | Order-grain fact, materialised zodat Silver-correcties automatisch propageren |
| `DATAMART.FCT_SALES_LINE` | MV (Liquid Clustering, in `dlt_datamart`) | Zwaarste tabel, profiteert van clustering |

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Catalog: {catalog}")

In [ ]:
# Volgorde — sales_line eerst (FCT_SALES_LINE MV in dlt_datamart leest hiervan),
# daarna de 9 dim-notebooks. Paden zijn relatief t.o.v. deze notebook.
view_notebooks = [
    "integration/sales_line",
    "datamart/dim_date",
    "datamart/dim_truck",
    "datamart/dim_location",
    "datamart/dim_customer",
    "datamart/dim_menu_item",
    "datamart/dim_currency",
    "datamart/dim_order_channel",
    "datamart/dim_shift",
    "datamart/dim_discount",
]

for nb in view_notebooks:
    dbutils.notebook.run(nb, 0, {"catalog": catalog})
    print(f"  applied: {nb}")

## Validatie

Lijst alle views in `INTEGRATION` + `DATAMART` om te bevestigen dat alles
aangemaakt is.

In [ ]:
# Expected uppercase object names applied by this orchestrator.
# DLT-managed objects (DW_/DWH_/DWQ_ in INTEGRATION, FCT_* in DATAMART) may
# also appear in SHOW VIEWS output but are not under apply_views' purview.
expected = {
    "INTEGRATION": ["SALES_LINE"],
    "DATAMART": [
        "DIM_DATE",
        "DIM_TRUCK",
        "DIM_LOCATION",
        "DIM_CUSTOMER",
        "DIM_MENU_ITEM",
        "DIM_CURRENCY",
        "DIM_ORDER_CHANNEL",
        "DIM_SHIFT",
        "DIM_DISCOUNT",
    ],
}

for schema, expected_views in expected.items():
    rows = spark.sql(f"SHOW VIEWS IN {catalog}.{schema}").collect()
    actual = {r["viewName"].upper() for r in rows}
    print(f"\n{catalog}.{schema} — {len(rows)} view(s):")
    for r in rows:
        print(f"  {r['viewName']}")
    missing = [v for v in expected_views if v not in actual]
    assert not missing, f"Missing expected views in {schema}: {missing}"